# Foundations

This notebook uses the educational estimators in `umd_ml`. Every table it builds is synthetic: a local NumPy Generator draws the rows, and nothing is downloaded. Install the package first with `python -m pip install -e ".[dev]"`, then run the cells from the repository root.


## Supervised learning

Supervised learning fits a rule from features `X` to a known label `y`, then asks how that rule behaves on rows it did not fit. Regression predicts a number. Classification predicts a category. The features are the inputs. The label is the thing we claim to predict.


In [ ]:
import numpy as np
import pandas as pd

from umd_ml import (
    LinearRegression,
    KNeighborsClassifier,
    NeuralNetwork,
    StandardScaler,
    accuracy,
    confusion_matrix,
    make_classification,
    make_regression,
    standardize_frames,
    train_test_split,
)

regression = make_regression(
    40,
    1,
    coefficients=np.array([1.5]),
    intercept=-0.5,
    noise=0.2,
    seed=5541,
)
print(regression.source)
print(regression.description)
regression.to_frame().head()


## A line, fit by gradient descent

The prediction is `x1 * slope + intercept`. Training starts at slope 0 and intercept 0, then steps against the gradient of mean squared error. `loss_history_` is that convergence history. The first entry is the loss before any update.


In [ ]:
model = LinearRegression(learning_rate=0.1, n_iterations=400)
model.fit(regression.X, regression.y)
print("slope", float(model.coef_[0]), "intercept", model.intercept_)
print("loss", float(model.loss_history_[0]), "->", float(model.loss_history_[-1]))


## Splitting, then scaling without leakage

Hold out test rows before you estimate anything you will reuse. The scaler below is fit on two training numbers whose mean is 2. The test numbers are 7 and 9. A scaler fit on all four rows would report a mean of 5. That is leakage: the test rows would have moved the transformation that is later scored.


In [ ]:
train = pd.DataFrame({"x1": [1.0, 3.0], "label": [0, 1]})
test = pd.DataFrame({"x1": [7.0, 9.0], "label": [0, 1]})
safe_train, safe_test, scaler = standardize_frames(train, test, ["x1"])
leaked = StandardScaler().fit(pd.concat([train, test])[["x1"]].to_numpy())
print("training mean", float(scaler.mean_[0]))
print("leaked mean", float(leaked.mean_[0]))
print(safe_test)


## Classification with k-nearest neighbors

Distance is Euclidean. The `k` nearest training labels vote. Equal distances keep the earlier training row. Equal votes keep the smallest label. The split below is stratified so each class appears on both sides. Accuracy is a reasonable summary here because the generator builds balanced classes. It would be a poor summary on a rare-event problem.


In [ ]:
blobs = make_classification(
    80,
    2,
    separation=2.5,
    noise=0.4,
    seed=5541,
)
print(blobs.source)
x_train, x_test, y_train, y_test = train_test_split(
    blobs.X,
    blobs.y,
    test_size=0.25,
    seed=5541,
    stratify=blobs.y,
)
knn = KNeighborsClassifier(k=5).fit(x_train, y_train)
predicted = knn.predict(x_test)
print("held-out accuracy", accuracy(y_test, predicted))
print(confusion_matrix(y_test, predicted, labels=np.array([0, 1])))


## One hidden layer

The network computes an affine map, applies ReLU or sigmoid, then computes a second affine map. Regression stops there and trains with mean squared error. `fit` restarts from `seed`, so running this cell twice prints the same loss history. The curve is the training loss on these rows. It is not a test score.


In [ ]:
curve = make_regression(
    60,
    1,
    coefficients=np.array([0.8]),
    intercept=0.1,
    noise=0.05,
    seed=5541,
)
network = NeuralNetwork(
    n_inputs=1,
    n_hidden=8,
    hidden_activation="relu",
    learning_rate=0.05,
    n_iterations=200,
    seed=5541,
)
network.fit(curve.X, curve.y)
history = network.loss_history_
print("parameters", network.n_parameters)
print("training loss", float(history[0]), "->", float(history[-1]))


## What a synthetic score does not say

These rows were drawn to make the arithmetic checkable. A high accuracy means the classifier found a gap the generator inserted. It does not say anything about people, and these models are not for decisions that affect health, money, work, or education. Seeds, float64, and the tie rules are what make a second run match the first. See `docs/responsible-ml.md` and the root `README.md` for the longer argument.
